# Homework 3: Data Analysis

This notebook focuses on comprehensive data analysis including:
- Exploratory Data Analysis (EDA)
- Statistical analysis
- Model performance analysis
- Feature importance analysis
- Results visualization and interpretation

## 1. Setup and Imports

In [ ]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Machine Learning libraries
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.inspection import permutation_importance

# Custom database module
import sys
sys.path.append('..')
from database.data_loader import DataLoader

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

## 2. Load Data and Model

In [ ]:
# Initialize data loader
loader = DataLoader()

# Load data and predictions from previous homeworks
# df = loader.load_csv('preprocessed_data.csv')
# predictions = loader.load_csv('predictions.csv')
# model = joblib.load('best_model.pkl')

# For demonstration, create sample data
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5, 
                           n_redundant=3, random_state=42)
df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])
df['target'] = y

print(f"Dataset shape: {df.shape}")
df.head()

## 3. Exploratory Data Analysis

### 3.1 Distribution Analysis

In [ ]:
# Plot distributions of numerical features
numerical_cols = df.select_dtypes(include=[np.number]).columns.drop('target')

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols):
    axes[idx].hist(df[col], bins=30, edgecolor='black')
    axes[idx].set_title(f'Distribution of {col}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

### 3.2 Correlation Analysis

In [ ]:
# Calculate correlation matrix
correlation_matrix = df.corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# Find highly correlated features
high_corr = np.where(np.abs(correlation_matrix) > 0.7)
high_corr_list = [(correlation_matrix.index[x], correlation_matrix.columns[y], 
                   correlation_matrix.iloc[x, y]) 
                  for x, y in zip(*high_corr) if x != y and x < y]

print("\nHighly correlated feature pairs:")
for feat1, feat2, corr in high_corr_list:
    print(f"{feat1} - {feat2}: {corr:.3f}")

### 3.3 Target Distribution

In [ ]:
# Analyze target variable
target_counts = df['target'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot
target_counts.plot(kind='bar', ax=axes[0], color=['skyblue', 'coral'])
axes[0].set_title('Target Variable Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%',
            colors=['skyblue', 'coral'], startangle=90)
axes[1].set_title('Target Variable Proportion')

plt.tight_layout()
plt.show()

print(f"\nClass balance: {target_counts.values / len(df)}")

## 4. Statistical Analysis

### 4.1 Feature Comparison by Target

In [ ]:
# Compare features across target classes
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols):
    for target_val in df['target'].unique():
        subset = df[df['target'] == target_val][col]
        axes[idx].hist(subset, alpha=0.5, label=f'Class {target_val}', bins=20)
    
    axes[idx].set_title(f'{col} by Target')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()

plt.tight_layout()
plt.show()

### 4.2 Statistical Tests

In [ ]:
# Perform t-tests for each feature
print("T-test results (difference between classes):")
print("="*60)

for col in numerical_cols:
    class_0 = df[df['target'] == 0][col]
    class_1 = df[df['target'] == 1][col]
    
    t_stat, p_value = stats.ttest_ind(class_0, class_1)
    
    significance = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
    print(f"{col:20s}: t-statistic = {t_stat:7.3f}, p-value = {p_value:.4f} {significance}")

## 5. Model Performance Analysis

In [ ]:
# Train a model for analysis
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

### 5.1 Confusion Matrix Analysis

In [ ]:
# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Calculate metrics from confusion matrix
tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")
print(f"\nSpecificity: {tn/(tn+fp):.3f}")
print(f"Sensitivity (Recall): {tp/(tp+fn):.3f}")

### 5.2 ROC Curve and AUC

In [ ]:
# Plot ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

### 5.3 Precision-Recall Curve

In [ ]:
# Plot Precision-Recall curve
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid(alpha=0.3)
plt.show()

## 6. Feature Importance Analysis

In [ ]:
# Get feature importances from Random Forest
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 5 most important features:")
print(feature_importance.head())

## 7. Summary and Insights

In [ ]:
# Generate summary report
print("="*70)
print("ANALYSIS SUMMARY")
print("="*70)

print("\n1. Dataset Overview:")
print(f"   - Total samples: {len(df)}")
print(f"   - Number of features: {len(X.columns)}")
print(f"   - Class balance: {df['target'].value_counts().to_dict()}")

print("\n2. Model Performance:")
print(f"   - Accuracy: {(y_pred == y_test).mean():.3f}")
print(f"   - AUC-ROC: {roc_auc:.3f}")

print("\n3. Top 3 Important Features:")
for idx, row in feature_importance.head(3).iterrows():
    print(f"   - {row['feature']}: {row['importance']:.4f}")

print("\n4. Recommendations:")
print("   - Review features with low importance for potential removal")
print("   - Consider collecting more data if class imbalance exists")
print("   - Explore feature engineering for highly correlated features")

print("\n" + "="*70)

## 8. Export Results

In [ ]:
# Save analysis results
# feature_importance.to_csv('feature_importance.csv', index=False)
# correlation_matrix.to_csv('correlation_matrix.csv')
print("Analysis complete!")